<a href="https://colab.research.google.com/github/naamasarshalom-art/segmentation_cellpose/blob/main/6_model2_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Model 2 — Morphology Classifier (Good vs Invaginated)

This notebook trains **Model 2**: a Logistic Regression classifier that distinguishes
morphologically normal nuclei (`good`) from invaginated ones (`invaginated`).

Model 2 is applied only to nuclei that passed **Model 1** quality control
(i.e., classified as `classifiable`, not `trash`).

**Input:** DINOv3 CLS embeddings computed in notebook `5_DINOv3_embeddings.ipynb`

**Pipeline:**
1. Load embeddings and filter to good (0) and invaginated (1) classes
2. Train / test split
3. Model selection: benchmark Logistic Regression, SVM, Random Forest, ExtraTrees
4. Cross-validation on training set (mean +/- std)
5. Train final Logistic Regression on full training set
6. Evaluate on held-out test set
7. Save the trained model

**Output:** `model2_lr.pkl` — trained Logistic Regression classifier

## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import numpy as np
import pandas as pd
import joblib

from sklearn.model_selection import (
    train_test_split, StratifiedKFold,
    cross_val_predict, cross_validate
)
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.metrics import (
    make_scorer, roc_auc_score, average_precision_score,
    precision_score, recall_score, f1_score,
    classification_report, confusion_matrix
)

## 2. Configuration

In [ ]:
# Input embeddings produced by 4_DINOv2_embeddings.ipynb
EMBEDDINGS_PATH = "/content/drive/MyDrive/model_nuc/embeddings_dino.npz"

# Directory where the trained model will be saved
MODEL_DIR = "/content/drive/MyDrive/model_nuc"

# Only good (0) and invaginated (1) are used for Model 2.
# Unclassifiable (2) and trash (3) are excluded.
CLASS_NAMES = {0: "good", 1: "invaginated"}

RANDOM_STATE = 42

## 3. Load Data & Filter to Good vs Invaginated

In [ ]:
data   = np.load(EMBEDDINGS_PATH, allow_pickle=True)
X_all  = data["X"]
labels = data["labels"]

# Keep only good (0) and invaginated (1)
mask = (labels == 0) | (labels == 1)
X = X_all[mask]
y = labels[mask]

print(f"Embedding shape: {X.shape}")
print("\nClass distribution:")
for cls, name in CLASS_NAMES.items():
    n = (y == cls).sum()
    print(f"  {name:>15}: {n:>4} samples ({100*n/len(y):.1f}%)")
print(f"  {'Total':>15}: {len(y):>4}")

## 4. Train / Test Split

20% held-out test set — not used until final evaluation.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y
)

print(f"Train: {len(X_train)} samples")
print(f"Test:  {len(X_test)} samples (held out)")

## 5. Model Selection — Benchmark

Compare four classifiers using 5-fold cross-validation on the training set.
DINOv2 embeddings are used directly without PCA or scaling.

In [ ]:
MODEL_DICT = {
    "LogReg": Pipeline([("model", LogisticRegression(
        penalty="l2", C=0.2, class_weight="balanced",
        max_iter=2000, random_state=RANDOM_STATE))]),

    "SVM (RBF)": Pipeline([("model", SVC(
        probability=True, class_weight="balanced",
        random_state=RANDOM_STATE))]),

    "RandomForest": Pipeline([("model", RandomForestClassifier(
        n_estimators=300, class_weight="balanced",
        random_state=RANDOM_STATE))]),

    "ExtraTrees": Pipeline([("model", ExtraTreesClassifier(
        n_estimators=300, class_weight="balanced",
        random_state=RANDOM_STATE))]),
}


def benchmark_models(X, y, model_dict, cv_splits=5):
    """Evaluate each model with stratified CV. Returns a summary DataFrame."""
    results = []
    cv = StratifiedKFold(n_splits=cv_splits, shuffle=True, random_state=RANDOM_STATE)

    for name, pipe in model_dict.items():
        y_prob = cross_val_predict(pipe, X, y, cv=cv, method="predict_proba")[:, 1]
        y_pred = (y_prob >= 0.5).astype(int)

        results.append({
            "Model":     name,
            "ROC AUC":   round(roc_auc_score(y, y_prob), 4),
            "PR AUC":    round(average_precision_score(y, y_prob), 4),
            "Recall":    round(recall_score(y, y_pred), 4),
            "Precision": round(precision_score(y, y_pred), 4),
            "F1":        round(f1_score(y, y_pred), 4),
        })

    return pd.DataFrame(results).sort_values("ROC AUC", ascending=False)


df_benchmark = benchmark_models(X_train, y_train, MODEL_DICT)
print(df_benchmark.to_string(index=False))

## 6. Cross-Validation on Training Set

5-fold stratified cross-validation for the final Logistic Regression.
Reports mean +/- std for all key metrics.

In [ ]:
clf_cv = LogisticRegression(
    penalty="l2", C=0.2,
    class_weight="balanced",
    max_iter=2000,
    random_state=RANDOM_STATE
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

scoring = {
    "roc_auc":   "roc_auc",
    "f1":        make_scorer(f1_score),
    "recall":    make_scorer(recall_score),
    "precision": make_scorer(precision_score),
    "accuracy":  "accuracy",
}

cv_results = cross_validate(
    clf_cv, X_train, y_train,
    cv=cv, scoring=scoring,
    return_train_score=False
)

print("=" * 48)
print("CROSS-VALIDATION RESULTS (5-fold, train set)")
print("=" * 48)
for metric in ["roc_auc", "f1", "recall", "precision", "accuracy"]:
    vals = cv_results[f"test_{metric}"]
    print(f"  {metric:<12}: {vals.mean():.4f} +/- {vals.std():.4f}")

## 7. Train Final Model & Evaluate on Test Set

Logistic Regression selected based on benchmark CV performance.
Trained on the full training set, evaluated once on the held-out test set.

In [ ]:
clf_final = LogisticRegression(
    penalty="l2", C=0.2,
    class_weight="balanced",
    max_iter=2000,
    random_state=RANDOM_STATE
)
clf_final.fit(X_train, y_train)

# Evaluate on held-out test set
y_test_prob = clf_final.predict_proba(X_test)[:, 1]
y_test_pred = (y_test_prob >= 0.5).astype(int)

print("=" * 48)
print("TEST SET RESULTS")
print("=" * 48)
print(f"  ROC AUC:   {roc_auc_score(y_test, y_test_prob):.4f}")
print(f"  PR AUC:    {average_precision_score(y_test, y_test_prob):.4f}")
print(f"  F1:        {f1_score(y_test, y_test_pred):.4f}")
print(f"  Recall:    {recall_score(y_test, y_test_pred):.4f}")
print(f"  Precision: {precision_score(y_test, y_test_pred):.4f}")
print()
print(classification_report(
    y_test, y_test_pred,
    target_names=["good", "invaginated"]
))

# Confusion matrix
cm = confusion_matrix(y_test, y_test_pred, labels=[0, 1])
tn, fp, fn, tp = cm.ravel()
cm_df = pd.DataFrame(
    [[f"TN ({tn})", f"FP ({fp})"],
     [f"FN ({fn})", f"TP ({tp})"]],
    index=["TRUE: good", "TRUE: invaginated"],
    columns=["PRED: good", "PRED: invaginated"]
)
print(cm_df)

## 8. Save Model

In [ ]:
# Save the trained Logistic Regression classifier.
# No scaler or PCA needed — DINOv2 CLS embeddings are passed directly.
save_path = f"{MODEL_DIR}/model2_lr.pkl"
joblib.dump(clf_final, save_path)
print(f"Model saved: {save_path}")